# MAP-Elite (Gaussian + Line, Boltzmann Mix) — Baseline

**Method:** MAP-Elites with `PSEMut` + `PSELine` (Boltzmann Mix adaptive allocation)  
**Seeds:** 2 (42, 123)  
**Task:** Ant-v5 Unidirectional Gait (4D Gait BD, 10×10×10×10 = 10000 bins)  
**Budget:** N_TOTAL × N_STEPS = 1e5

In [ ]:
# ============================================================
# Cell 1: Setup — mount Drive, clone repo, load SSLVE modules
# Run this cell once at the start of each Colab session.
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!rm -rf org && git clone --filter=blob:none --sparse https://github.com/yugoguy/org.git
!cd org && git sparse-checkout set dev/SSLVE && git checkout Latent-Variable-Evolution
!pip install cma gymnasium[mujoco] -q

import sys, os, glob
sys.path.insert(0, 'org/dev/SSLVE')
for f in sorted(glob.glob('org/dev/SSLVE/*.py')):
    %run {f}

print('Setup complete.')

Mounted at /content/drive
Cloning into 'org'...
remote: Enumerating objects: 1903, done.
remote: Counting objects: 100% (222/222), done.
remote: Compressing objects: 100% (177/177), done.
remote: Total 1903 (delta 144), reused 47 (delta 42), pack-reused 1681 (from 1)
Receiving objects: 100% (1903/1903), 531.72 KiB | 3.67 MiB/s, done.
Resolving deltas: 100% (803/803), done.
remote: Enumerating objects: 1, done.
remote: Total 1 (delta 0), reused 0 (delta 0), pack-reused 1 (from 1)
Receiving objects: 100% (1/1), 255 bytes | 255.00 KiB/s, done.
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 18 (delta 5), reused 1 (delta 1), pack-reused 6 (from 1)
Receiving objects: 100% (18/18), 72.87 KiB | 2.51 MiB/s, done.
Resolving deltas: 100% (6/6), done.
Branch 'Latent-Variable-Evolution' set up to track remote branch 'Latent-Variable-Evolution' from 'origin'.
Switched to a new branch 'Latent-Variable

In [ ]:
# ============================================================
# Cell 2: Method Hyperparameters
# ============================================================
import numpy as np
import random
import torch

# --- Architecture ---
ARCHITECTURE     = [27, 64, 64, 8]  #@param
OUTPUT_ACTIVATION = 'tanh'  #@param {type:"string"}
MAX_STEPS        = 500   #@param {type:"integer"}
N_EPISODES       = 3     #@param {type:"integer"}
CTRL_COST_WEIGHT = 0.5   #@param {type:"number"}

# --- Gait BD Grid ---
BIN_SIZES        = [10, 10, 10, 10]  #@param

# --- Fitness ---
TOP_K            = 1     #@param {type:"integer"}
MAX_FITNESS      = 500.0 #@param {type:"number"}
GREEDY_MEM       = True  #@param {type:"boolean"}

# --- Mutation sigma ---
PSE_MUT_SIGMA    = 0.05  #@param {type:"number"}

# --- Boltzmann Mix ---
EMA_ALPHA        = 0.25  #@param {type:"number"}
TEMPERATURE      = 10    #@param {type:"number"}
MIN_PROPORTION   = 0.05  #@param {type:"number"}

# --- Search budget ---
N_TOTAL          = 500   #@param {type:"integer"}
N_STEPS          = 200   #@param {type:"integer"}

# --- Checkpoint base directory (saved to Google Drive) ---
CKPT_BASE_DIR    = '/content/drive/MyDrive/shared_ckpts/ant_gait/Ant_MAPElite_GaussLine_BoltzMix/'  #@param {type:"string"}

print(f'Method: MAP-Elite (Gaussian + Line, Boltzmann Mix)')
print(f'Architecture: {ARCHITECTURE}, Output: {OUTPUT_ACTIVATION}')
print(f'Gait BD: {BIN_SIZES} ({np.prod(BIN_SIZES)} bins)')
print(f'PSE_MUT_SIGMA={PSE_MUT_SIGMA}')
print(f'EMA_ALPHA={EMA_ALPHA}, TEMPERATURE={TEMPERATURE}, MIN_PROPORTION={MIN_PROPORTION}')
print(f'N_TOTAL={N_TOTAL}, N_STEPS={N_STEPS}')
print(f'Checkpoints will be saved to: {CKPT_BASE_DIR}')

Method: MAP-Elite (Gaussian + Line, Boltzmann Mix)
Architecture: [27, 64, 64, 8], Output: tanh
Gait BD: [10, 10, 10, 10] (10000 bins)
PSE_MUT_SIGMA=0.05
EMA_ALPHA=0.25, TEMPERATURE=10, MIN_PROPORTION=0.05
N_TOTAL=500, N_STEPS=200
Checkpoints will be saved to: /content/drive/MyDrive/shared_ckpts/ant_gait/Ant_MAPElite_GaussLine_BoltzMix/


In [ ]:
# ============================================================
# Cell 3: Run All Experiments (2 seeds)
# Each run saves a checkpoint + plots immediately after finishing.
# ============================================================

SEEDS = [42, 123]

fitness_fn = lambda info: -info['forward_sum']

all_results = []

weight_dim = MLP_Agent(ARCHITECTURE, output_activation=OUTPUT_ACTIVATION).get_weight_dim()
print(f'Weight dim: {weight_dim}')

for seed in SEEDS:
    print(f'\n========== seed={seed} ==========')

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    collector = AntOmniCollector(
        max_steps=MAX_STEPS,
        n_episodes=N_EPISODES,
        ctrl_cost_weight=CTRL_COST_WEIGHT,
        seed=seed,
    )
    bd = AntGaitBD(bin_sizes=BIN_SIZES)
    bm = MAPElitesBM(behavior_descriptor=bd, fitness_fn=fitness_fn, top_k=TOP_K, max_fitness=MAX_FITNESS)

    sp = BoltzmannMix(
        agent_class=MLP_Agent,
        architecture=ARCHITECTURE,
        agent_kwargs={'output_activation': OUTPUT_ACTIVATION},
        operators=[PSEMut(sigma=PSE_MUT_SIGMA, greedy_mem=GREEDY_MEM), PSELine(greedy_mem=GREEDY_MEM)],
        n_total=N_TOTAL,
        warmup_threshold=999999,
        ema_alpha=EMA_ALPHA,
        temperature=TEMPERATURE,
        min_proportion=MIN_PROPORTION,
    )

    me = MAPElite(search_phase=sp, collector=collector, behavior_matching=bm)
    me.run(n_steps=N_STEPS)

    # Save checkpoint immediately after this run
    ckpt_path = os.path.join(
        CKPT_BASE_DIR,
        f'MAPElite_GaussLine_BoltzMix_seed{seed}/'
    )
    save_checkpoint(ckpt_path, bm, me.history, sp=sp)
    print(f'Checkpoint saved: {ckpt_path}')

    # Save plots into checkpoint folder
    me.plot_history(save_path=ckpt_path + 'plot_history.png')
    sp.plot_allocation(save_path=ckpt_path + 'plot_allocation.png')
    print(f'Plots saved to: {ckpt_path}')

    # Force Drive sync after each run
    from google.colab import drive
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print('Drive re-synced.')

    # Print results
    f_min, f_mean, f_max = bm.fitness_stats()
    qd  = bm.qd_score()
    cov = bm.coverage()
    print(f'QD-score: {qd:.4f} | Coverage: {cov:.4f} | Best fitness: {f_min:.6f}')

    all_results.append({
        'seed': seed,
        'qd_score': qd, 'coverage': cov,
    })

print('\n====== All runs complete ======')

NameError: name 'MLP_Agent' is not defined

In [ ]:
# ============================================================
# Cell 4: Results Summary — mean +- std across 2 seeds
# Copy these values into the shared results table.
# ============================================================
import pandas as pd

df = pd.DataFrame(all_results)

print('Method: MAP-Elite (Gaussian + Line, Boltzmann Mix)')
print(f'Hyperparameters: PSE_MUT_SIGMA={PSE_MUT_SIGMA}, EMA_ALPHA={EMA_ALPHA}, TEMPERATURE={TEMPERATURE}')
print(f'N_TOTAL={N_TOTAL}, N_STEPS={N_STEPS}')
print()

qd_mean, qd_std = df['qd_score'].mean(), df['qd_score'].std(ddof=1)
cov_mean, cov_std = df['coverage'].mean(), df['coverage'].std(ddof=1)

print(f'{"Metric":<25} {"Value":>25}')
print('-' * 52)
print(f'{"QD-score":<25} {f"{qd_mean:.4f} +- {qd_std:.4f}":>25}')
print(f'{"Behavior Coverage":<25} {f"{cov_mean:.4f} +- {cov_std:.4f}":>25}')